# ZS601 full-mesh 1cm, scale 0.5: four-view smoke ONLY

Uses a newly sampled complete Blender mesh surface cloud, with training-only color observations. Formal training still uses the original3cm cloud.

Uses standard 3-NN RMS scale with multiplier 0.5 and requested opacity 0.999999.
Zero training steps. View IDs: 3001, 3238, 3301, 3478. No full-run cell is included.
The user must inspect these images before authorizing the complete 200 views.

Compare only with the identical mesh1cm cloud at scale1.0; export no-coverage and low-coverage masks locally. Full200 remains gated.


In [1]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, platform
import torch
ROOT=Path('/content/zs601-mesh1cm-scale05-v004')
PKG=ROOT/'source/gaussian-splatting-lidar-init'
INPUT=ROOT/'input'
RUN=ROOT/'run'
RUN.mkdir(exist_ok=False)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def save(name,obj):
    with (RUN/name).open('x') as f:json.dump(obj,f,indent=2,allow_nan=False)
def run(command,log):
    print('$',' '.join(map(str,command)),flush=True)
    with (RUN/log).open('x') as f:
        p=subprocess.Popen(list(map(str,command)),cwd=PKG,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:f.write(line);f.flush();print(line,end='',flush=True)
        rc=p.wait()
    if rc:raise RuntimeError(f'{log} failed: {rc}')
env=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0),
    optimization_steps=0,init_scale_factor=0.5,opacity=0.999999,
    source_commit='4c7186e363f14050c4977bb192f12ed237112764',
    nvidia_smi=subprocess.check_output(['nvidia-smi'],text=True))
save('environment.json',env)
print(json.dumps(env,indent=2))


{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "NVIDIA L4",
  "capability": [
    8,
    9
  ],
  "optimization_steps": 0,
  "init_scale_factor": 0.5,
  "opacity": 0.999999,
  "source_commit": "4c7186e363f14050c4977bb192f12ed237112764",
  "nvidia_smi": "Mon Sep 21 14:12:11 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                                         |                        |               MIG M. |\n|=========================================+========================+======================|\n|   0  NVIDIA L4              

In [2]:
source=json.loads((ROOT/'source_manifest.json').read_text())
assert source['code_commit']==env['source_commit']
for r in source['files']:assert sha(ROOT/'source'/r['path'])==r['sha256'],r['path']
inputs=json.loads((INPUT/'input_manifest.json').read_text())
for r in inputs['files']:assert sha(INPUT/r['path'])==r['sha256'],r['path']
assert inputs['view_ids']==[3001,3238,3301,3478]
save('identity_verified.json',dict(source_commit=source['code_commit'],source_files=len(source['files']),
    input_files=len(inputs['files']),point_cloud_sha256=sha(INPUT/'points_mesh_1cm.ply')))
print('Source and smoke inputs verified.')


Source and smoke inputs verified.


In [3]:
assert sys.version_info[:2]==(3,13) and torch.__version__=='2.11.0+cu128'
assert torch.cuda.get_device_capability(0)==(8,9)
run([sys.executable,'-m','pip','install','plyfile==1.1.3'],'install_python.log')
WHEELS=ROOT/'wheels'
wheels=sorted(WHEELS.glob('*.whl'))
assert len(wheels)==2
run([sys.executable,'-m','pip','install','--no-index','--no-deps']+wheels,'install_cuda.log')
run([sys.executable,'check_contract.py','--sparse',INPUT/'sparse/0'],'check_contract.log')
save('reused_wheels.json',[dict(name=p.name,bytes=p.stat().st_size,sha256=sha(p)) for p in wheels])
save('installed_environment.json',dict(pip_freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)))


$ /usr/bin/python3 -m pip install plyfile==1.1.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.0 MB/s eta 0:00:00


$ /usr/bin/python3 -m pip install --no-index --no-deps /content/zs601-mesh1cm-scale05-v004/wheels/diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl /content/zs601-mesh1cm-scale05-v004/wheels/simple_knn-1.0.0-cp313-cp313-linux_x86_64.whl


Processing /content/zs601-mesh1cm-scale05-v004/wheels/diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl


Processing /content/zs601-mesh1cm-scale05-v004/wheels/simple_knn-1.0.0-cp313-cp313-linux_x86_64.whl


$ /usr/bin/python3 check_contract.py --sparse /content/zs601-mesh1cm-scale05-v004/input/sparse/0


{'camera_count': 200, 'max_projection_error_px': 2.0463630789890885e-12, 'uint16_mm_roundtrip': True}


In [4]:
# SMOKE ONLY. Explicit whitelist cannot render any of the other 196 views.
smoke=RUN/'smoke'
run([sys.executable,'render_from_sparse_v4.py','--point-cloud',INPUT/'points_mesh_1cm.ply',
     '--sparse',INPUT/'sparse/0','--output',smoke,'--opacity','0.999999',
     '--init-scale-factor','0.5','--view-ids','3001,3238,3301,3478'],'smoke_render.log')
run([sys.executable,'verify_outputs.py','--point-cloud',INPUT/'points_mesh_1cm.ply',
     '--sparse',INPUT/'sparse/0','--output',smoke,'--expected-views','4','--ground-truth',INPUT/'gt'],'smoke_verify.log')
report=json.loads((smoke/'verification.json').read_text())
assert report['views']==4 and report['optimization_steps']==0
save('SMOKE_COMPLETE.json',dict(status='SMOKE_VERIFIED_AWAITING_USER_REVIEW',views=4,optimization_steps=0,
    init_scale_factor=0.5,formal_200_authorized=False,source_commit=env['source_commit']))
print(json.dumps(report,indent=2))
print('STOP: full 200 views require user review and confirmation.')


$ /usr/bin/python3 render_from_sparse_v4.py --point-cloud /content/zs601-mesh1cm-scale05-v004/input/points_mesh_1cm.ply --sparse /content/zs601-mesh1cm-scale05-v004/input/sparse/0 --output /content/zs601-mesh1cm-scale05-v004/run/smoke --opacity 0.999999 --init-scale-factor 0.5 --view-ids 3001,3238,3301,3478


Number of points at initialisation :  7180820


{"index": 1, "image_id": 3001, "name": "003001.png", "visible_gaussians": 1216103, "coverage_alpha95": 0.9999788470216606, "coverage_alpha50": 1.0, "seconds": 1.609140157699585}


{"index": 4, "image_id": 3478, "name": "003478.png", "visible_gaussians": 83024, "coverage_alpha95": 0.9831199232851986, "coverage_alpha50": 1.0, "seconds": 1.4212160110473633}


RENDER_COMPLETE


...

  },


  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or optimized 3DGS claims",


  "ssim": "RGB [0,1], Gaussian 11x11 sigma1.5, C1=0.01^2 C2=0.03^2, fully valid windows"


}


{
  "status": "VERIFIED_INITIALIZATION_NOT_TRAINED",
  "views": 4,
  "points": 7180820,
  "optimization_steps": 0,
  "exact_xyz": true,
  "max_sh0_rgb_error": 7.807039748009004e-08,
  "opacity_decoded_min": 0.9999989867206773,
  "opacity_decoded_max": 0.9999989867206773,
  "png_count": 28,
  "camera_pose_and_intrinsics_exact": true,
  "means_over_views": {
    "alpha95_coverage": 0.9951453914711192,
    "depth_coverage": 1.0,
    "native_psnr_gt_valid_db": 24.47380696002451,
    "straight_psnr_gt_valid_db": 24.49350128415759,
    "straight_psnr_covered_db": 24.489462265340876,
    "native_ssim_gt_valid": 0.8311030268669128,
    "straight_ssim_covered": 0.8325856029987335,
    "ssim_covered_window_fraction": 0.9094945143276173,
    "depth_mae_mm": 14.700897503060844,
    "depth_rmse_mm": 59.140286824756686,
    "depth_absrel": 0.00659661104557314,
    "depth_eval_coverage": 0.9973964209160651
  },
  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or o

The four-view smoke is finished. Inspect native black RGB and straight RGB,
with matched v001 factor-0.5 images and Blender GT. Full rendering is gated by
the user's next confirmation; this notebook never launches it.

Compare only with the identical mesh1cm cloud at scale1.0; export no-coverage and low-coverage masks locally. Full200 remains gated.
